# skdr-eval — Quickstart

Offline policy evaluation in one screen. This notebook generates
synthetic logs, evaluates two candidate sklearn policies with the
DR / SNDR estimators, and prints the stakeholder summary.

👉 [Open in Colab](https://colab.research.google.com/github/dgenio/skdr-eval/blob/main/examples/notebooks/01_quickstart.ipynb)

## 1. Install

In [ ]:
%pip install --quiet skdr-eval scikit-learn

## 2. Build logs and candidate models

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

import skdr_eval

logs, items, _ = skdr_eval.make_synth_logs(n=2000, n_ops=4, seed=0)
skdr_eval.validate_logs(logs)

models = {
    "RF": RandomForestRegressor(n_estimators=80, random_state=0),
    "HGB": HistGradientBoostingRegressor(max_iter=80, random_state=0),
}
len(logs), list(items)

## 3. Evaluate

In [ ]:
artifact = skdr_eval.evaluate_sklearn_models(
    logs=logs,
    models=models,
    fit_models=True,
    n_splits=2,
    random_state=0,
    policy_train="pre_split",
    policy_train_frac=0.8,
)
artifact.report[
    ["model", "estimator", "V_hat", "ESS", "match_rate", "support_health"]
].round(4)

## 4. Trust signals

`artifact.warnings` aggregates support-health codes (ESS / overlap /
Pareto-k / calibration) into a single pass/caution/high-risk verdict
per (model, estimator).

In [ ]:
artifact.warnings

## 5. Stakeholder card

`save_card` writes a self-contained HTML page for the named model.

In [ ]:
import pathlib

pathlib.Path("artifacts").mkdir(exist_ok=True)
artifact.save_card("artifacts/quickstart_card.html", "HGB")

## Next steps

- Swap `make_synth_logs` for your own DataFrame; the schema is
  documented in [`README`](https://github.com/dgenio/skdr-eval#readme).
- For pairwise / autoscaling decisions, see the next notebook.
- For domain-flavored use cases, see notebooks 03–05.
